In [1]:
import plots
import datasets
from models.models_mlp import track_activation_patterns
import torch
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import tqdm

device = "cuda:0"
training_trajectories = plots.load_training_trajectories(
    32,
    plots.ExperimentMetadata("wmlp", "mnist", list(range(1, 25)))
)
dataset = datasets.get_mnist_dataset()

In [2]:
# trajectory = training_trajectories[0]
# data_example = dataset[0][0]

dataset_by_label = {i: [] for i in range(10)}
for data, label in dataset:
    dataset_by_label[label].append(data)

activation_patterns_by_label_and_lin = {
    label: { lin_index: None for lin_index in range(3)} for label in range(10)
}

BATCH_SIZE = 1024

# TODO also consider across models / across trajectory of one model
# models = [trajectory[-1] for trajectory in training_trajectories]
models = training_trajectories[0]

for label, data in tqdm.tqdm(dataset_by_label.items(), leave=False):
    loader = torch.utils.data.DataLoader(data, BATCH_SIZE)
    # batches = [data[i:i+BATCH_SIZE] for i in range(0, len(data), BATCH_SIZE)]
    patterns_by_model_and_lin = []
    for model_index, model in enumerate(models):
        patterns_by_lin = [[] for lin_index in range(3)]
        for batch in tqdm.tqdm(loader, leave=False):
            track_activation_patterns(model.activation, -1e-5)
            model.forward(batch.to(device))
            for lin_index in range(3):
                patterns_by_lin[lin_index].append(model.activation.tracked_activation_patterns[lin_index])
        patterns_by_model_and_lin.append(patterns_by_lin)
    
    for lin_index in range(3):
        activation_patterns_by_label_and_lin[label][lin_index] = torch.stack([
            torch.cat(patterns_by_model_and_lin[model_index][lin_index])
            for model_index in range(len(models))
        ])


# for model in trajectory:
#     for data_example, _ in dataset[:500]:
#         track_activation_patterns(model.activation, 1e-5)
#         model.forward(data_example.to(device))
#         activation_patterns_by_sample.append([model.activation.tracked_activation_patterns[lin_index].reshape(-1) for lin_index in range(3)])

# # lin_index = 1
# # picked_indices = [torch.randperm(np.prod(lin.weight.shape))[:512] for lin in trajectory[0].lins]

# activation_patterns = np.stack(
#     [ for lin_index in range(3) for model in trajectory],
#     axis=1
# )

# px.imshow(activation_patterns.T)

In [5]:
px.imshow(torch.cat([
    activation_patterns_by_label_and_lin[label][1].float()[:, 0, :] != activation_patterns_by_label_and_lin[label][1].float()[0, 0, :]
    # .mean(axis=1).std(axis=0)
    for label in range(10)]).cpu(),
    height=1000
)

In [65]:
px.imshow(torch.stack([activation_patterns_by_label_and_lin[label][1].float().mean(axis=1).std(axis=0) for label in range(10)]).cpu())

### MLPs, whole dataset (by label)

In [4]:
[activation_patterns_by_label_and_lin[label][1].float().mean(axis=1).std(axis=0).mean().item() for label in range(10)]

[0.6017736196517944,
 0.6114147901535034,
 0.5366837382316589,
 0.5484124422073364,
 0.5821152925491333,
 0.5415846109390259,
 0.5775159597396851,
 0.5492413640022278,
 0.5821123123168945,
 0.5897523760795593]

### MLPs, single sample per label

In [14]:
[activation_patterns_by_label_and_lin[label][1].float().std(axis=0).mean().item() for label in range(10)]

[0.8711001873016357,
 0.8869152665138245,
 0.9113717675209045,
 0.9159983992576599,
 0.8945870399475098,
 0.9123300909996033,
 0.9171348810195923,
 0.8922896385192871,
 0.9023229479789734,
 0.9130550622940063]

### WMLPs, whole dataset (per label)

In [7]:
[activation_patterns_by_label_and_lin[label][1].float().std(axis=0).mean().item() for label in range(10)]

[0.4395493268966675,
 0.4727466404438019,
 0.43498167395591736,
 0.4524068832397461,
 0.45427191257476807,
 0.4432467818260193,
 0.43399834632873535,
 0.4523767828941345,
 0.4570692181587219,
 0.47756433486938477]

### MLPs, single sample per label

In [9]:
[activation_patterns_by_label_and_lin[label][1].float().std(axis=0).mean().item() for label in range(10)]

[0.43128475546836853,
 0.4721371531486511,
 0.45157450437545776,
 0.40476304292678833,
 0.4274233877658844,
 0.45563697814941406,
 0.4860648512840271,
 0.43501925468444824,
 0.4380204975605011,
 0.4755851626396179]

Conclusions?

- W-MLPs have lower stddev. between models (still not very low, but significantly lower than MLPs).
- surprisingly (does it tell us anything?) W-MLPs sttdevs are almost the same between whole-dataset and single example (??)